# Embeddings multimodales: CLIP y DINOv2

<a href="https://colab.research.google.com/github/noe-tec/machine-learning/blob/main/activities/activity_multimodal_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Un *embedding* es un vector que representa algo de modo que las cosas parecidas quedan cerca. En esta actividad se comparan dos modelos que aprenden embeddings de imágenes de formas distintas:

- **CLIP**, que coloca imágenes y textos en un mismo espacio.
- **DINOv2**, que aprende embeddings de imágenes sin usar texto ni etiquetas.

En cada celda marcada con `TODO` hay que reemplazar `None` por el código correspondiente; la celda siguiente comprueba el resultado. Se recomienda activar un entorno con GPU (*Entorno de ejecución → Cambiar tipo de entorno*), aunque la libreta también funciona en CPU.

In [ ]:
!pip install -q open_clip_torch

import gzip, io, tarfile, time, urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)

## CLIP y DINOv2

**CLIP** tiene dos *encoders*, uno para imágenes y otro para textos, entrenados para que una imagen y su descripción queden cerca en un mismo espacio. Se usará OpenCLIP ViT-B/32, una versión de CLIP entrenada con datos públicos (LAION-400M).

**DINOv2** solo tiene un encoder de imágenes. Se entrenó sin texto ni etiquetas: dos recortes distintos de la misma foto deben producir la misma salida. Se usará su versión más pequeña, DINOv2-small.

La siguiente celda descarga ambos modelos.

In [ ]:
import open_clip
from transformers import AutoModel

clip, _, preprocesar_clip = open_clip.create_model_and_transforms(
    "ViT-B-32-quickgelu", pretrained="laion400m_e32")
tokenizar = open_clip.get_tokenizer("ViT-B-32-quickgelu")
clip = clip.to(dispositivo).eval()

dino = AutoModel.from_pretrained("facebook/dinov2-small").to(dispositivo).eval()
print("Modelos listos.")

La galería de imágenes es Imagenette, un subconjunto de ImageNet con 10 clases. Se toman 20 imágenes por clase del conjunto de validación (200 imágenes en total).

In [ ]:
CLASES = {"n01440764": "tench", "n02102040": "English springer", "n02979186": "cassette player",
          "n03000684": "chain saw", "n03028079": "church", "n03394916": "French horn",
          "n03417042": "garbage truck", "n03425413": "gas pump", "n03445777": "golf ball",
          "n03888257": "parachute"}

def indexar_imagenette(destino="imagenette"):
    """Descarga Imagenette (160 px) y agrupa las imágenes de validación por carpeta."""
    url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
    raiz = Path(destino)
    marca = raiz / ".descarga_completa"              # evita usar una extracción interrumpida
    if not marca.exists():
        with urllib.request.urlopen(url) as r:
            tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz").extractall(raiz, filter="data")
        marca.touch()
    grupos = {}
    for ruta in raiz.rglob("*"):
        if ruta.suffix.lower() in (".jpeg", ".jpg", ".png") and any(x.startswith("val") for x in ruta.parts):
            grupos.setdefault(ruta.parent.name, []).append(ruta)
    return {carpeta: sorted(rutas) for carpeta, rutas in grupos.items()}

def cargar_galeria(por_clase=20):
    """Descarga Imagenette (160 px) y regresa imágenes y etiquetas."""
    grupos = indexar_imagenette()
    imagenes, etiquetas = [], []
    for wnid, nombre in CLASES.items():
        if len(grupos.get(wnid, [])) < por_clase:
            raise RuntimeError(f"Solo se encontraron {len(grupos.get(wnid, []))} imágenes de {wnid}. "
                               f"Carpetas encontradas: {sorted(grupos)[:12]}")
        for ruta in grupos[wnid][:por_clase]:
            imagenes.append(Image.open(ruta).convert("RGB"))
            etiquetas.append(nombre)
    return imagenes, np.array(etiquetas)

imagenes, etiquetas = cargar_galeria()
nombres_clases = list(CLASES.values())
print(len(imagenes), "imágenes;", len(nombres_clases), "clases")

fig, ejes = plt.subplots(2, 5, figsize=(12, 5))
for eje, i in zip(ejes.flat, range(0, len(imagenes), 20)):
    eje.imshow(imagenes[i]); eje.set_title(etiquetas[i], fontsize=10); eje.axis("off")
plt.tight_layout(); plt.show()

Las funciones siguientes calculan los embeddings y los normalizan a longitud 1, de modo que la similitud coseno sea un producto punto. Para DINOv2 se usa el vector del token de clase (CLS).

In [ ]:
@torch.no_grad()
def emb_clip_imagenes(imgs, lote=64):
    salida = []
    for i in range(0, len(imgs), lote):
        x = torch.stack([preprocesar_clip(im) for im in imgs[i:i + lote]]).to(dispositivo)
        salida.append(clip.encode_image(x).float().cpu())
    e = torch.cat(salida).numpy()
    return e / np.linalg.norm(e, axis=1, keepdims=True)

@torch.no_grad()
def emb_clip_textos(textos):
    e = clip.encode_text(tokenizar(textos).to(dispositivo)).float().cpu().numpy()
    return e / np.linalg.norm(e, axis=1, keepdims=True)

MEDIA, DESV = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

@torch.no_grad()
def emb_dino(imgs, lote=64):
    salida = []
    for i in range(0, len(imgs), lote):
        x = np.stack([(np.asarray(im.resize((224, 224), Image.BICUBIC), np.float32) / 255 - MEDIA) / DESV
                      for im in imgs[i:i + lote]])
        x = torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2).to(dispositivo)
        salida.append(dino(pixel_values=x).last_hidden_state[:, 0].cpu())
    e = torch.cat(salida).numpy()
    return e / np.linalg.norm(e, axis=1, keepdims=True)

inicio = time.time()
G_clip, G_dino = emb_clip_imagenes(imagenes), emb_dino(imagenes)
print(f"CLIP: {G_clip.shape}, DINOv2: {G_dino.shape}, en {time.time() - inicio:.0f} s")

### Clasificación *zero-shot* con CLIP

CLIP puede clasificar sin reentrenar: se escribe un texto por clase («a photo of a {clase}»), se calcula la similitud de la imagen con cada texto y se aplica un softmax. CLIP multiplica las similitudes por una escala aprendida (`clip.logit_scale.exp()`, cercana a 100) antes del softmax.

In [ ]:
# TODO 4: regresa una matriz de probabilidades [n_imagenes, n_clases].
# 1) T = emb_clip_textos([f"a photo of a {c}" for c in clases])
# 2) logits = escala * (G @ T.T)
# 3) softmax por filas (resta el máximo de cada fila antes de exponenciar).
def zero_shot(G, clases):
    escala = clip.logit_scale.exp().item()
    return None

In [ ]:
# Comprobación del TODO 4
def comprobar_4():
    P = zero_shot(G_clip, nombres_clases)
    if P is None:
        print("Pendiente: zero_shot todavía regresa None.")
        return
    if P.shape != (len(imagenes), len(nombres_clases)) or not np.allclose(P.sum(axis=1), 1, atol=1e-4):
        print("Revisa: la forma debe ser (200, 10) y cada fila debe sumar 1.")
        return
    exactitud = (np.array(nombres_clases)[P.argmax(axis=1)] == etiquetas).mean()
    print(f"Correcto. Exactitud zero-shot en la galería: {exactitud:.1%}, sin entrenar con estas imágenes.")
comprobar_4()

### Búsqueda de imágenes parecidas

Con embeddings normalizados, buscar las imágenes más parecidas a una consulta es ordenar los productos punto de la consulta contra toda la galería. Se compararán los vecinos que encuentra CLIP con los que encuentra DINOv2.

In [ ]:
# TODO 5: regresa los índices de las k imágenes de la galería G más parecidas a la consulta q,
# de mayor a menor similitud y sin incluir el índice `excluir` (la propia consulta).
def buscar(q, G, k=5, excluir=None):
    return None

In [ ]:
# Comprobación del TODO 5
def comprobar_5():
    r = buscar(G_dino[0], G_dino, k=5, excluir=0)
    if r is None:
        print("Pendiente: buscar todavía regresa None.")
        return
    if len(r) != 5 or 0 in r:
        print("Revisa: deben ser 5 índices y no debe aparecer la propia consulta.")
        return
    sims = G_dino[r] @ G_dino[0]
    if np.any(np.diff(sims) > 1e-6):
        print("Revisa: los resultados deben ir de mayor a menor similitud.")
        return
    def precision(G):
        return np.mean([np.mean(etiquetas[buscar(G[i], G, 5, i)] == etiquetas[i]) for i in range(len(G))])
    print(f"Correcto. Precisión de los 5 vecinos (misma clase): CLIP {precision(G_clip):.1%}, "
          f"DINOv2 {precision(G_dino):.1%}")
comprobar_5()

In [ ]:
def comparar(i, k=5):
    """Muestra la consulta i y sus k vecinos según CLIP y según DINOv2."""
    fig, ejes = plt.subplots(2, k + 1, figsize=(2.2 * (k + 1), 5))
    for fila, (nombre, G) in enumerate([("CLIP", G_clip), ("DINOv2", G_dino)]):
        ejes[fila, 0].imshow(imagenes[i]); ejes[fila, 0].set_title(f"consulta\n({nombre})", fontsize=10)
        for c, j in enumerate(buscar(G[i], G, k, i), start=1):
            ejes[fila, c].imshow(imagenes[j]); ejes[fila, c].set_title(etiquetas[j], fontsize=9)
    for e in ejes.flat: e.axis("off")
    plt.tight_layout(); plt.show()

def buscar_texto(texto, k=5):
    """Búsqueda por texto: solo CLIP puede hacerla."""
    idx = buscar(emb_clip_textos([texto])[0], G_clip, k)
    fig, ejes = plt.subplots(1, k, figsize=(2.2 * k, 2.6))
    for e, j in zip(ejes, idx): e.imshow(imagenes[j]); e.set_title(etiquetas[j], fontsize=9); e.axis("off")
    plt.suptitle(f"«{texto}»"); plt.tight_layout(); plt.show()

comparar(41)
buscar_texto("a red truck on the street")

### Reto en equipo

1. Prueben `comparar(i)` con distintas imágenes de la galería (índices del 0 al 199) y encuentren al menos un caso donde CLIP acierte y DINOv2 no, y otro donde ocurra lo contrario.
2. Prueben `buscar_texto(...)` con descripciones que no sean nombres de clase, por ejemplo «a person holding a musical instrument».
3. Opcional: suban una foto propia con la celda siguiente y busquen sus vecinos con ambos modelos.

Cada equipo presentará su mejor ejemplo.

In [ ]:
# Opcional: subir una foto propia (solo en Colab).
# from google.colab import files
# nombre = next(iter(files.upload()))
# foto = Image.open(nombre).convert("RGB")
# for modelo, q, G in [("CLIP", emb_clip_imagenes([foto])[0], G_clip), ("DINOv2", emb_dino([foto])[0], G_dino)]:
#     print(modelo, [etiquetas[j] for j in buscar(q, G)])

## Preguntas

Respondan en esta celda, en una o dos oraciones cada una.

1. En el reto, ¿en qué tipo de consultas ganó CLIP y en cuáles DINOv2? ¿A qué lo atribuyen?
2. ¿Por qué DINOv2 no puede buscar imágenes a partir de un texto?

**Respuestas:**

1.
2.